# BioChannel Notebook 3 — Drug Response Predictor

**Process drug-response data and generate dose-response outputs.**

This notebook is part of the **BioChannel** Kaggle hackathon project. It is designed to be run inside Kaggle Notebooks and then reused by the Streamlit dashboard.

## How to use this notebook in BioChannel

1. Attach the relevant Kaggle dataset using **Add Data** in the Kaggle notebook sidebar.
2. Run the notebook end-to-end.
3. Save processed outputs into `/kaggle/working/processed/`.
4. Copy the exported CSV/JSON files into the BioChannel Streamlit app under `data/processed/`.
5. Reuse the helper functions in the app modules: `data_loader.py`, `preprocessing.py`, `simulation.py`, `info_theory.py`, and `prediction.py`.

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.metrics import mutual_info_score
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

np.random.seed(42)


In [ ]:
from pathlib import Path

INPUT_DIR = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')
PROCESSED_DIR = WORKING_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)

print('Input datasets available:')
for p in INPUT_DIR.glob('*'):
    print(' -', p)

## Dataset

Recommended datasets:

- GDSC Drug Response: `ghazaleh/gdsc-drug-response`
- Alternative: `cdc/drug-response-prediction`

Science mapping:

`drug + dose + cell line → survival/apoptosis/resistance response`

BioChannel treats the drug as an intervention signal and the response as the cellular decision output.

In [ ]:
csv_files = list(INPUT_DIR.rglob('*.csv'))
for f in csv_files[:20]: print(f)

In [ ]:
def load_drug_response_or_demo(csv_files):
    for f in csv_files:
        try:
            df = pd.read_csv(f)
            cols = [c.lower() for c in df.columns]
            has_drug = any('drug' in c or 'compound' in c for c in cols)
            has_response = any('ic50' in c or 'response' in c or 'auc' in c or 'viability' in c for c in cols)
            if has_drug and has_response and len(df) > 20:
                print('Using:', f)
                return df
        except Exception:
            pass
    print('No suitable dataset found. Creating demo drug-response data.')
    cell_lines = ['A549','MCF7','HCT116','PC9','Resistant-like']
    drugs = ['Drug-A','Drug-B','Drug-C','Combo-A+B']
    rows = []
    for cell in cell_lines:
        for drug in drugs:
            sensitivity = np.random.uniform(0.25, 0.85)
            if 'Resistant' in cell: sensitivity -= 0.25
            if 'Combo' in drug: sensitivity += 0.15
            rows.append({'cell_line': cell, 'drug': drug, 'sensitivity': np.clip(sensitivity,0,1), 'ic50_proxy': np.clip(1-sensitivity+np.random.normal(0,0.05),0,1)})
    return pd.DataFrame(rows)

df = load_drug_response_or_demo(csv_files)
df.head()

In [ ]:
# Infer columns. Adjust manually for your Kaggle dataset if needed.
cols = list(df.columns)
lower_map = {c: c.lower() for c in cols}
cell_col = next((c for c in cols if 'cell' in c.lower() or 'line' in c.lower()), None)
drug_col = next((c for c in cols if 'drug' in c.lower() or 'compound' in c.lower()), None)
response_col = next((c for c in cols if 'ic50' in c.lower() or 'response' in c.lower() or 'auc' in c.lower() or 'viability' in c.lower() or 'sensitivity' in c.lower()), None)

if cell_col is None: cell_col = cols[0]
if drug_col is None: drug_col = cols[1]
if response_col is None:
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    response_col = numeric_cols[0]

work = df[[cell_col, drug_col, response_col]].dropna().copy()
work.columns = ['cell_line','drug','raw_response']
work['raw_response'] = pd.to_numeric(work['raw_response'], errors='coerce')
work = work.dropna()

# Normalize response. If response is IC50-like, lower can mean stronger sensitivity. The UI can explain this caveat.
r = work['raw_response']
work['response_norm'] = (r - r.min()) / (r.max() - r.min() + 1e-9)
work['sensitivity_score'] = 1 - work['response_norm']

work.to_csv(PROCESSED_DIR / 'drug_response_clean.csv', index=False)
work.head()

In [ ]:
# Lightweight predictor for dashboard demo.
le_cell = LabelEncoder()
le_drug = LabelEncoder()
work['cell_encoded'] = le_cell.fit_transform(work['cell_line'].astype(str))
work['drug_encoded'] = le_drug.fit_transform(work['drug'].astype(str))

X = work[['cell_encoded','drug_encoded']]
y = work['sensitivity_score']

if len(work) >= 30:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
else:
    X_train, X_test, y_train, y_test = X, X, y, y

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print('MAE:', mean_absolute_error(y_test, pred))

summary = work.groupby(['cell_line','drug'], as_index=False)['sensitivity_score'].mean()
summary.to_csv(PROCESSED_DIR / 'drug_response_summary.csv', index=False)
summary.head()

In [ ]:
example = summary.iloc[0]
sensitivity = float(example['sensitivity_score'])
doses = np.linspace(0,1,80)
survival = 1 / (1 + np.exp(8 * (doses - sensitivity)))
apoptosis = 1 - survival

plt.figure(figsize=(7,4))
plt.plot(doses, survival, label='Survival probability')
plt.plot(doses, apoptosis, label='Apoptosis probability')
plt.title(f'Dose-response: {example.cell_line} + {example.drug}')
plt.xlabel('Dose')
plt.ylabel('Probability')
plt.legend()
plt.show()

## Streamlit integration

Use:

- `drug_response_clean.csv` → raw selector data
- `drug_response_summary.csv` → fast UI dropdowns and default response curves

Gemma prompt should include cell line, drug, dose, sensitivity score, resistance pressure, and whether the selected configuration resembles sensitivity, partial response, or resistance.